Script for training Resnet (18, 50, 101), two protocols, retinotopic by integrating a log-polar transformation during re-training,
image maps without transformation but a circular mask is applied to the image. 

Test the 3 data sets:

* full (Imagenet full image) (done)
* square (the smallest square containing the bounding box) (done)
* focus (best point of fixation after a saccade)

TODO: make a second learning to double epochs and show it saturates

In [5]:
from retinotopy import *
welcome()

-----------------------------------------------------------------------------------
On date 2025-06-20, Running learning on host Ahsoka with device cpu, pytorch==2.2.2
-----------------------------------------------------------------------------------
Welcome on macOS-15.5-x86_64-i386-64bit


In [7]:
datetag

'2025-06-20'

In [11]:
for i in range(6, 20 ): print(i)

6
7
8
9
10
11
12
13
14
15
16
17
18
19


# Transfer learning on the full and bbox datasets in Cartesian and retinotopic coordinates

Doing this for different ResNet flavors : 18, 50 and 101

In [ ]:
# Training and saving the networks
for data_set_type in data_set_types:
# for data_set_type in ['bbox']:
    
    print(50*'=')
    print(f'{data_set_type=}')
    print(50*'-')
    fig, ax = plt.subplots(1, 1, figsize=(fig_width, fig_width/phi), subplotpars=subplotpars)
        
    for model_name, lw in  zip(['resnet18', 'resnet50', 'resnet101'], [1, 2, 4]): #, 
        for do_polar, color in zip([True, False], ['b', 'r']):
            # setting parameters
            args = Params()
            args.root  = f'{DATAROOT}/Imagenet_{data_set_type}' # Directory containing images to perform the training
            args.do_polar = do_polar
            
            print(50*'.')
            model_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '.pt'
            json_filename = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '.json'

            df_train = None
            if os.path.isfile(json_filename):
                print(f"Load JSON from pre-trained resnet {json_filename}")
                df_train = pd.read_json(json_filename, orient='index')
                print(f"{model_filename}: accuracy = {df_train['avg_acc_val'][-5:].mean():.3f}")

            if not os.path.isfile(model_filename):
                # the learning has not been started yet
                touch(model_filename + '.lock') # we want to have a file let's lock it
                model_path = None if data_set_type == 'full' else get_filename(data_cache, datetag, 'full', model_name, do_polar) + '.pt' 
            else:
                # the learning has been started, we want to load the model
                model_path = get_filename(data_cache, datetag, data_set_type, model_name, do_polar) + '.pt' 
                
            if os.path.isfile(model_filename + '.lock') or df_train is None:
                # get the architecture of the network
                print(f"Loading {model_path=}")
                model_retrain = load_model(model_name=model_name, model_path=model_path, do_scratch=args.do_scratch, do_circular=args.do_polar).to(device)

                # we need to train the model or finish a training that already started
                print(f"Training model {model_name}, file= {model_filename} - image_size={args.image_size}")
                since = time.time()
                model_retrain, df_train = train_model(args, model_retrain, dataloaders=datasets_transforms(args), df_train=df_train, model_filename=model_filename)
                elapsed_time = time.time() - since
                print(f"Training completed in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")

                print()     


            if not(df_train is None):
                # import lmfit
                from lmfit import Parameters, Model

                x = df_train['total_image'].values
                y = df_train['avg_acc_val'].values

                # Define the function
                def model(x, acc_max, acc_min, x_50) :
                    return acc_max - (acc_max-acc_min) * np.exp( - x / x_50)

                # Init the sigmoid model as an lmfit Model object 
                mod = Model(model)
                pars = Parameters()
                # Add the initial parameters guesses
                pars.add_many(('acc_max', np.max(y), True,  0.0, 1.),
                            ('x_50', np.max(x)/20, True, 1, np.max(x)),
                            ('acc_min', 1/1000, True,  0.0, 1.))
                # And fit using least-square minimization
                out = mod.fit(y, pars, x=x, nan_policy='omit', max_nfev = 3000)
                # Print the result
                result = f"fitted max accuracy= {out.best_values['acc_max']:.2f}, speed= {out.best_values['x_50']:.1f}"
                print(result)

                df_train_roll = df_train.rolling(window=5, min_periods=1, center=False).mean()
                ax = df_train_roll.plot(x='total_image', y='avg_acc', 
                                    c=color, ls='dashed', lw=lw,
                                    grid=True, ax=ax, label='TRAIN: ' + json_filename.strip(data_cache + '/').strip('.json'))    
                ax = df_train_roll.plot(x='total_image', y='avg_acc_val', 
                                    c=color, lw=lw,
                                    grid=True, ax=ax, label='VAL: ' + json_filename.strip(data_cache + '/').strip('.json') + '-' + result)   

            # the model has been trained, we can remove the lock file     
            if os.path.isfile(model_filename + '.lock'): os.remove(model_filename + '.lock')

    ax.set_ylim(.15, .85)            
    ax.set_ylim(.05, .95)            
    ax.set_yscale('logit')
    plt.show()
    print(50*'=')